In [1]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69

In [2]:
from langchain_core.documents import Document

In [3]:
from langchain_community.document_loaders.text import TextLoader

loader=TextLoader("/content/Python.txt",encoding="utf-8")

In [4]:
document=loader.load()

In [5]:
document

[Document(metadata={'source': '/content/Python.txt'}, page_content='\ufeffPython is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automati

In [ ]:
#PDF Data
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader=PyPDFLoader("/content/Research1.pdf")
# document=pdf_loader.load()
# document

In [ ]:
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader=PyMuPDFLoader("/content/Research1.pdf")
# document=pdf_loader.load()
# document

## Ingestion Pipeline

In [6]:
#Data => Documents

import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [7]:
def load_all_pdfs():
    folder_path = "/content/data"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [8]:
all_pdf_documents=load_all_pdfs()

total pdfs: 2
total pages: 32


In [9]:
#Chunks
!pip install langchain_text_splitters

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [11]:
chunks= split_docs(all_pdf_documents)

In [12]:
len(chunks)

321

##Embedding

In [13]:
from sentence_transformers import SentenceTransformer

In [14]:
class EmbeddingManager:
  def __init__(self,model_name="all-MiniLM-L6-v2"):
    self.model_name=model_name
    print("loading model....",self.model_name)
    self.model= SentenceTransformer(self.model_name)
    print("embedding dimensions=",self.model.get_sentence_embedding_dimension())

  def generate_embeddings(self,text):
    embeddings=self.model.encode(text, show_progress_bar=True)
    print("embeddings shape: ",embeddings.shape)
    return embeddings

In [15]:
embedding_manager=EmbeddingManager()

loading model.... all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding dimensions= 384


/tmp/ipykernel_1879/2131639521.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=",self.model.get_sentence_embedding_dimension())


## Vector Store

In [16]:
import chromadb
import uuid

In [17]:
class VectorStoreManager:
  def __init__(self,persist_directory="data/vector_store",collection_name="pdf_documents"):
    self.persist_directory=persist_directory
    self.collection_name=collection_name
    self.collection=None
    self.client=None
    self._initialize_store()

  def _initialize_store(self):
    os.makedirs(self.persist_directory,exist_ok=True)
    #create a client
    self.client= chromadb.PersistentClient(path=self.persist_directory)

    #create the collection
    self.collection=self.client.get_or_create_collection(
        name=self.collection_name,
        metadata={"description":"vector store collection for pdf embeddings in RAG"}
    )

    print("initialized the vector store with collection: ",self.collection_name)
    print("docs in collection: ",self.collection.count())

  def add_documents(self, documents, embeddings):
    if len(documents)!= len(embeddings):
      raise ValueError("num of documents does not match num of embeddings")

    ids=[]
    all_metadata=[]
    documents_content=[]
    embeddings_list=[]

    for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
      doc_id=f"doc_{uuid.uuid4()}"
      ids.append(doc_id)

      metadata= dict(doc.metadata)
      metadata["doc_index"]=i
      metadata["content_length"]= len(doc.page_content)
      all_metadata.append(metadata)

      documents_content.append(doc.page_content)

      embeddings_list.append(embedding.tolist())

      self.collection.add(
          ids=ids,
          metadatas=all_metadata,
          documents=documents_content,
          embeddings=embeddings_list
      )

    print("total documents added in vector store=",len(documents_content))
    print("docs in collection:",self.collection.count())



In [18]:
vector_store=VectorStoreManager()

initialized the vector store with collection:  pdf_documents
docs in collection:  0


In [19]:
texts= [doc.page_content for doc in chunks]

embedding= embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embeddings shape:  (321, 384)
total documents added in vector store= 321
docs in collection: 321


## Retrieval Pipeline

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
class RAGRetriever:
  def __init__(self, embedding_manager, vector_store):
    self.embedding_manager=embedding_manager
    self.vector_store=vector_store

  def retrieve(self, query, top_k=5, score_threshold=0.0):
    #Query Embedding
    query_embeddings= self.embedding_manager.generate_embeddings([query])[0]

    #Semantic Search
    results= self.vector_store.collection.query(
        query_embeddings=[query_embeddings.tolist()],
        n_results=top_k
    )

    #Cosine Similarity
    retrieved_docs=[]
    if results["documents"] and results["documents"][0]:
      ids= results["ids"][0]
      metadatas= results["metadatas"][0]
      documents=results["documents"][0]
      distances=results["distances"][0]

      for i ,(doc_id, metadata, document, distance) in enumerate(zip(ids , metadatas, documents, distances)):
        similarity_score= 1-distance
        if similarity_score>= score_threshold:
          retrieved_docs.append(
              {
                  "id": doc_id,
                  "document": document,
                  "metadata":metadata,
                  "distance":distance,
                  "similarity_score": similarity_score,
                  "rank": i+1
              }
          )
        print(f"retrieved {len(retrieved_docs)} documents")

    else:
      print("no documents found")

    return retrieved_docs

In [22]:
rag_retriever=RAGRetriever(embedding_manager, vector_store)

In [23]:
rag_retriever.retrieve("What is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape:  (1, 384)
retrieved 1 documents
retrieved 2 documents
retrieved 3 documents
retrieved 4 documents
retrieved 5 documents


[{'id': 'doc_2c2c835e-c887-4501-b4af-575e476efea9',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'producer': 'pdfTeX-1.40.25',
   'page': 0,
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'total_pages': 21,
   'page_label': '1',
   'source': '/content/data/research2.pdf',
   'author': '',
   'moddate': '2024-03-28T00:54:45+00:00',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'subject': '',
   'content_length': 288,
   'trapped': '/False',
   'doc_index': 11,
   'creator': 'LaTeX with hyperref',
   'keywords': '',
   'title': ''},
  'distance': 0.46291497349739075,
  'similarity_score': 0.5370850265026093,
  'rank': 1},
 {'id': 'doc_b62ed

## Integrate with LLMs

### OpenAI-GPT

In [ ]:
API_KEY_OPENAI= # Paste your own api key here

In [30]:
!pip install langchain-openai

In [31]:
from langchain_openai import ChatOpenAI

llm= ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-5.4",
    temperature=0.1,
    max_tokens=1024
)

In [32]:
#Generate our retrieval augmented output
def generate_output(query,retriever,llm,top_k=3):
  results= retriever.retrieve(query, top_k)

  context="\n".join(doc["document"] for doc in results) if results else ""

  if not context:
    print("We not found no relevant context for the given query")

  # prompt->context+query
  prompt= f""" use given context to generate the answer for the query
               Context : {context}
               Query: {query} """

  response=llm.invoke(prompt) # expecting a string as prompt
  return response.content

In [33]:
answer= generate_output("What is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape:  (1, 384)
retrieved 1 documents
retrieved 2 documents
retrieved 3 documents


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************h4cA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

### Groq

In [ ]:
API_KEY_GROQ= #Paste your own api key here

In [ ]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.3 MB/s eta 0:00:00


In [ ]:
from langchain_groq import ChatGroq

llm= ChatGroq(
    groq_api_key=API_KEY_GROQ,
    model="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024
)


In [ ]:
#Generate our retrieval augmented output
def generate_output(query,retriever,llm,top_k=3):
  results= retriever.retrieve(query, top_k)

  context="\n".join(doc["document"] for doc in results) if results else ""

  if not context:
    print("We not found no relevant context for the given query")

  # prompt->context+query
  prompt= f""" use given context to generate the answer for the query
               Context : {context}
               Query: {query} """

  response=llm.invoke([prompt.format(context=context,query=query)]) # expecting a list as prompt
  return response.content

In [ ]:
answer= generate_output("What is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape:  (1, 384)
retrieved 1 documents
retrieved 2 documents
retrieved 3 documents


In [ ]:
print(answer)

<think>
Okay, the user is asking, "What is RAG?" Let me start by recalling the context provided. The context mentions a survey paper that reviews state-of-the-art RAG methods, their evolution through paradigms like naive RAG, and effective frameworks. It also talks about evaluation methods, tasks, datasets, and future directions.

First, I need to define RAG. From the context, RAG stands for Retrieval-Augmented Generation. The main idea is combining retrieval from external knowledge sources with a generative model, probably a large language model (LLM). The survey discusses how RAG integrates retrieval and generation, which helps in providing up-to-date and accurate information by fetching relevant documents before generating a response.

The context mentions different paradigms of RAG, like naive RAG and effective frameworks. I should explain that RAG has evolved over time, starting with basic approaches and moving towards more sophisticated methods. The survey also covers evaluation 